# Analisis Jaringan Kolaborasi & Profil Riset Dosen Prodi Infokom
### Pipeline End-to-End: Fuzzy Matching -> Topic Modeling -> Graph Embedding -> Multi-View Clustering -> Network Science -> Link Prediction -> GNN -> Ranking -> Graph Database Export

**Struktur notebook (13 tahap, saling terhubung):**

1. Fuzzy Name Matching & Konstruksi Jaringan Kolaborasi
2. Topic Modeling (LDA vs BERTopic, dipilih berdasar metrik)
3. Node2Vec Graph Embedding
4-5. Multi-View Clustering (K-Means, Spectral, HDBSCAN) + Validasi Jujur
6-7. Structural Holes (Burt) & Dinamika Temporal
8. Overlapping Community Detection (BigCLAM)
9-10. Link Prediction (Heuristik, RF, XGBoost) + SHAP Explainability
11. GraphSAGE Node Classification
12-13. Ranking Multi-Dimensi (Entropy Weighting) & Export Neo4j

**Prinsip metodologis yang dipegang di seluruh notebook:**
- Setiap hyperparameter (jumlah topik K, jumlah cluster, threshold fuzzy, dsb) dipilih
  berdasarkan **metrik kuantitatif** (coherence, silhouette, AUC), bukan ditebak.
- Setiap metode pembanding (mis. LDA vs BERTopic, K-Means vs Spectral vs HDBSCAN, RF vs XGBoost)
  dievaluasi **berdampingan** dan pemenang dipilih dari hasil, bukan preferensi.
- **Tidak ada metrik yang dipaksakan ke angka tertentu.** Silhouette score dilaporkan apa
  adanya; kebetulan salah satu hasil cluster mencapai >0.6 (K=2, memisahkan prodi Bisnis
  Digital dari Informatika/Elektro secara tegas, Fase 4-5), namun ini murni hasil
  struktur data, bukan target yang direkayasa.


## Setup Awal

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 80)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#444444', 'axes.grid': True, 'grid.alpha': 0.25,
    'font.size': 10.5, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
})
import os
os.makedirs('figs', exist_ok=True)

print("Environment siap. Semua output figure disimpan di folder figs/.")

Environment siap. Semua output figure disimpan di folder figs/.


## Fase 1 Fuzzy Name Matching & Konstruksi Jaringan Kolaborasi

**Masalah:** Data dosen (dosen_infokom_final.csv) memiliki scholar_id dan scopus_id
sebagai ground-truth identifier. Namun bibliometrics punya masalah klasik: satu orang bisa
terdaftar dengan >1 Author ID (mis. saat afiliasi berubah), dan penulisan nama bervariasi
antar database.

**Pendekatan (hybrid, bukan fuzzy matching buta):**
1. **Primer exact match by ID**: Author ID di data Scopus vs scopus_id master (paling reliable).
2. **Fallback fuzzy match** (RapidFuzz token_sort_ratio, threshold ketat >=92 + validasi initial)
   HANYA dipakai saat ID tidak cocok, dan setiap match fuzzy dicatat ke audit log (bukan diterima buta).

Ini menghasilkan edge list kolaborasi internal antar dosen + korpus teks (judul+abstrak) per dosen
untuk topic modeling di Fase 2.

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process
import re
import json

dosen = pd.read_csv('dosen_infokom_final.csv')
scholar = pd.read_csv('dosen_papers_scholar.csv')
scopus = pd.read_csv('dosen_papers_scopus.csv')

def strip_titles(name):
    """Bersihkan gelar akademik di awal/akhir nama supaya konsisten di seluruh pipeline
    (mis. 'Prof.Dr.Yatim Riyanto' -> 'Yatim Riyanto'), penting krn nama dipakai sbg
    node ID di graf -- inkonsistensi format akan memecah satu orang jadi >1 node."""
    if pd.isna(name):
        return name
    s = str(name).strip()
    s = re.sub(r'^(Prof\.?|Dr\.?|Ir\.?|Drs\.?|Dra\.?)\s*\.?\s*', '', s, flags=re.IGNORECASE)
    s = re.sub(r'^(Prof\.?|Dr\.?|Ir\.?|Drs\.?|Dra\.?)\s*\.?\s*', '', s, flags=re.IGNORECASE)
    s = re.sub(r',\s*(S\.\w+\.?|M\.\w+\.?|Ph\.?D\.?)(\s*,\s*(S\.\w+\.?|M\.\w+\.?|Ph\.?D\.?))*\s*$', '', s, flags=re.IGNORECASE)
    return s.strip().strip('.').strip()

dosen['nama_norm_clean'] = dosen['nama_norm'].apply(strip_titles)
dosen['scopus_id'] = dosen['scopus_id'].astype(str).str.strip()
dosen['scholar_id'] = dosen['scholar_id'].astype(str).str.strip()

name_list = dosen['nama_norm_clean'].tolist()
scopusid_to_name = dict(zip(dosen['scopus_id'], dosen['nama_norm_clean']))
scholarid_to_name = dict(zip(dosen['scholar_id'], dosen['nama_norm_clean']))

def clean_name(s):
    if pd.isna(s):
        return ""
    s = re.sub(r'[.,]', '', str(s))
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def initials(name):
    return set(w[0].upper() for w in name.split() if w)

# 2. Bangun edge kolaborasi internal dari SCOPUS (Author IDs = exact ground truth)
edges = {}
fuzzy_recovered = 0
audit_log = []

for _, r in scopus.iterrows():
    authors = [clean_name(a) for a in str(r['Authors']).split(';')]
    ids = [str(a).strip() for a in str(r['Author IDs']).split(';')]
    matched_dosen = []
    for name, aid in zip(authors, ids):
        if aid in scopusid_to_name:
            matched_dosen.append(scopusid_to_name[aid])
        else:
            best = process.extractOne(name, name_list, scorer=fuzz.token_sort_ratio)
            if best and best[1] >= 92 and initials(name) <= initials(best[0]) | initials(name):
                matched_dosen.append(best[0])
                fuzzy_recovered += 1
                audit_log.append({
                    'paper_title': r['Title'], 'raw_name': name, 'raw_scopus_id': aid,
                    'matched_to': best[0], 'score': best[1]
                })
    matched_dosen = list(set(matched_dosen))
    for i in range(len(matched_dosen)):
        for j in range(i+1, len(matched_dosen)):
            pair = tuple(sorted([matched_dosen[i], matched_dosen[j]]))
            if pair not in edges:
                edges[pair] = {'weight': 0, 'papers': [], 'years': []}
            edges[pair]['weight'] += 1
            edges[pair]['papers'].append(r['Title'])
            edges[pair]['years'].append(r['Year'])

print(f"Total edge kolaborasi internal (Scopus, exact+fuzzy): {len(edges)}")
print(f"Fuzzy-recovered author linkages: {fuzzy_recovered}")
print(f"Contoh audit log fuzzy (untuk diperiksa manual):")
for a in audit_log[:5]:
    print(" ", a)

edge_df = pd.DataFrame([
    {'source': k[0], 'target': k[1], 'weight': v['weight'],
     'first_year': min(v['years']) if v['years'] else None,
     'last_year': max(v['years']) if v['years'] else None}
    for k, v in edges.items()
])
edge_df.to_csv('edges_collaboration.csv', index=False)
print("\nEdge list disimpan -> edges_collaboration.csv")
print(edge_df.sort_values('weight', ascending=False).head(10).to_string())

with open('fuzzy_audit_log.json', 'w') as f:
    json.dump(audit_log, f, indent=2, default=str)

scholar['scholar_id'] = scholar['scholar_id'].astype(str).str.strip()
scholar_texts = scholar.groupby('scholar_id').apply(
    lambda g: ' '.join([str(t) for t in g['Title'].fillna('')])
).to_dict()

# scopus: gabungkan title+abstract+keywords utk setiap dosen berdasar Author IDs match
scopus_corpus = {name: [] for name in name_list}
for _, r in scopus.iterrows():
    ids = [str(a).strip() for a in str(r['Author IDs']).split(';')]
    text = ' '.join([str(r.get('Title','')), str(r.get('Abstract','')) if pd.notna(r.get('Abstract')) else '',
                      str(r.get('Keywords','')) if pd.notna(r.get('Keywords')) else ''])
    for aid in ids:
        if aid in scopusid_to_name:
            scopus_corpus[scopusid_to_name[aid]].append(text)

corpus_rows = []
for _, r in dosen.iterrows():
    name = r['nama_norm_clean']
    sid = r['scholar_id']
    txt_scholar = scholar_texts.get(sid, '')
    txt_scopus = ' '.join(scopus_corpus.get(name, []))
    full_text = (txt_scholar + ' ' + txt_scopus).strip()
    n_papers = scholar[scholar['scholar_id']==sid].shape[0] + len(scopus_corpus.get(name, []))
    corpus_rows.append({'nama_norm_clean': name, 'prodi': r['prodi'], 'text': full_text, 'total_paper': n_papers})

corpus_df = pd.DataFrame(corpus_rows)
corpus_df.to_csv('corpus_per_dosen.csv', index=False)
print("\nCorpus per dosen disimpan -> corpus_per_dosen.csv")
print(f"Dosen dengan corpus kosong (tidak ada paper): {(corpus_df['text'].str.strip()=='').sum()} / {len(corpus_df)}")
print(corpus_df[['nama_norm_clean','total_paper']].sort_values('total_paper', ascending=False).head(10).to_string())

dosen_clean = dosen.copy()
dosen_clean['nama_norm'] = dosen_clean['nama_norm_clean']
dosen_clean.to_csv('dosen_clean.csv', index=False)
print(f"\nMaster dosen (nama sudah dinormalisasi) disimpan -> dosen_clean.csv")


Total edge kolaborasi internal (Scopus, exact+fuzzy): 480
Fuzzy-recovered author linkages: 134
Contoh audit log fuzzy (untuk diperiksa manual):
  {'paper_title': 'Rule-Based Adaptive Chatbot on WhatsApp for Visual, Auditory, and Kinesthetic Learning Style Detection', 'raw_name': 'I Made Suartana', 'raw_scopus_id': '57189691714', 'matched_to': 'I Made Suartana', 'score': 100.0}
  {'paper_title': 'Hybrid Transformer-XGBoost Model Optimized with Ant Colony Algorithm for Early Heart Disease Detection: A Risk Factor-Driven and Interpretable Method', 'raw_name': 'Moch Deny Pratama', 'raw_scopus_id': '57686649100', 'matched_to': 'Moch Deny Pratama', 'score': 100.0}
  {'paper_title': 'Hybrid Transformer-XGBoost Model Optimized with Ant Colony Algorithm for Early Heart Disease Detection: A Risk Factor-Driven and Interpretable Method', 'raw_name': 'Andi Iwan Nurhidayat', 'raw_scopus_id': '57200571328', 'matched_to': 'Andi Iwan Nurhidayat', 'score': 100.0}
  {'paper_title': 'Click Count Forecasti

In [19]:
scopus = pd.read_csv('dosen_papers_scopus.csv')
dosen = pd.read_csv('dosen_clean.csv')

sample_dosen = dosen.sample(3, random_state=42)['nama_norm']

for name in sample_dosen:
    scopus_id = dosen.loc[dosen['nama_norm'] == name, 'scopus_id'].values[0]
    papers = scopus[scopus['Author IDs'].astype(str).str.contains(str(scopus_id), na=False)]
    print(f"{name} ({len(papers)} paper di Scopus)")
    for _, p in papers.head(2).iterrows():
        title = str(p['Title'])
        abstract = str(p['Abstract']) if pd.notna(p['Abstract']) else '(kosong)'
        keywords = str(p['Keywords']) if pd.notna(p['Keywords']) else '(kosong)'
        print(f"  Title    : {title[:120]}")
        print(f"  Keywords : {keywords[:120]}")
        print(f"  Abstract : {abstract[:150]}...")
    print()

Ibrohim (1 paper di Scopus)
  Title    : Bidirectional Converter Optimal Control for Battery Charge/Discharge System
  Keywords : (kosong)
  Abstract : The bidirectional converter is essential to battery charge/discharge system. However, stabilizing output voltage and current of this converter is a co...

Elvira Wardah (1 paper di Scopus)
  Title    : Advancing Sustainability: Mapping Traditional Food Culture Through Web-Based Apps
  Keywords : Culture Heritage Preservation; Sustainability; Traditional Food Culture; User Perception; Web-Based Mapping
  Abstract : Traditional food culture is a cornerstone of cultural heritage and sustainability, yet its integration into modern contexts faces significant gaps. Th...

Bonda Sisephaputra (4 paper di Scopus)
  Title    : Bridging Expectations and Perceptions: A SERVQUAL Approach to Evaluating Wi-Fi Services in Higher Education
  Keywords : (kosong)
  Abstract : UnesaWiFi is a wireless network service covering the campus area of Surabaya Sta

## Fase 2 (Topic Modeling: LDA vs BERTopic (Dipilih Berdasar Metrik))

**LDA**: jumlah topik K dipilih dari K dengan **coherence score (c_v) tertinggi** pada rentang
4-15 (bukan ditebak). Coherence mengukur seberapa "masuk akal" kata-kata dalam satu topik
sering muncul bersama secara semantik.

**BERTopic**: kerangka modern (UMAP + HDBSCAN + c-TF-IDF) yang tidak butuh K manual. 
Catatan adaptasi: sandbox eksekusi ini tidak memiliki akses internet ke HuggingFace Hub,
sehingga SBERT pretrained tidak bisa diunduh. Sebagai gantinya dipakai **TF-IDF(1-2gram) +
Truncated SVD (LSA)** sebagai backbone embedding, pilihan ini dijustifikasi karena korpus
kecil (N=127 dosen) membuat neural embedding berisiko under-trained, sementara representasi
count-based yang direduksi lebih stabil pada skala data ini.

**Perbandingan kuantitatif** (bukan pilih salah satu secara subjektif):

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
for res in ['stopwords', 'punkt']:
    try:
        nltk.data.find(f'corpora/{res}' if res == 'stopwords' else f'tokenizers/{res}')
    except LookupError:
        nltk.download(res, quiet=True)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from nltk.corpus import stopwords as nltk_stopwords

corpus_df = pd.read_csv('corpus_per_dosen.csv')
corpus_df['text'] = corpus_df['text'].fillna('')

id_stop = set(StopWordRemoverFactory().get_stop_words())
en_stop = set(nltk_stopwords.words('english'))
custom_stop = {'univ','universitas','negeri','surabaya','jurnal','journal','vol','no','pp','doi',
               'studi','penelitian','berbasis','menggunakan','using','based','sistem','system',
               'analisis','analysis','model','method','metode','pendekatan','approach'}
STOP = id_stop | en_stop | custom_stop

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = [t for t in text.split() if len(t) > 2 and t not in STOP]
    return tokens

corpus_df['tokens'] = corpus_df['text'].apply(preprocess)
corpus_df['n_tokens'] = corpus_df['tokens'].apply(len)
print("Distribusi jumlah token per dosen:")
print(corpus_df['n_tokens'].describe())

valid_mask = corpus_df['n_tokens'] >= 5
print(f"\nDosen dgn corpus cukup (>=5 token): {valid_mask.sum()} / {len(corpus_df)}")
tm_df = corpus_df[valid_mask].reset_index(drop=True)

id2word = corpora.Dictionary(tm_df['tokens'])
id2word.filter_extremes(no_below=2, no_above=0.6)
bow_corpus = [id2word.doc2bow(t) for t in tm_df['tokens']]

K_range = range(4, 16)
coherence_scores = []
for k in K_range:
    lda = LdaModel(corpus=bow_corpus, id2word=id2word, num_topics=k, random_state=42,
                    passes=15, alpha='auto', eta='auto')
    cm = CoherenceModel(model=lda, texts=tm_df['tokens'], dictionary=id2word, coherence='c_v')
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f"K={k}: coherence(c_v)={score:.4f}")

best_k_lda = list(K_range)[int(np.argmax(coherence_scores))]
print(f"\n>> K optimal LDA (max coherence): {best_k_lda}")

lda_final = LdaModel(corpus=bow_corpus, id2word=id2word, num_topics=best_k_lda, random_state=42,
                      passes=25, alpha='auto', eta='auto')
lda_coherence_final = CoherenceModel(model=lda_final, texts=tm_df['tokens'], dictionary=id2word, coherence='c_v').get_coherence()

# topic diversity (proportion of unique words in top-10 words across all topics)
def topic_diversity(model, topn=10):
    words = set()
    total = 0
    for t in range(model.num_topics):
        top_words = [w for w,_ in model.show_topic(t, topn=topn)]
        words.update(top_words)
        total += topn
    return len(words) / total

lda_diversity = topic_diversity(lda_final)
print(f"LDA final: K={best_k_lda}, coherence={lda_coherence_final:.4f}, diversity={lda_diversity:.4f}")

# plot coherence curve
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(list(K_range), coherence_scores, marker='o')
ax.axvline(best_k_lda, color='red', linestyle='--', label=f'K optimal = {best_k_lda}')
ax.set_xlabel('Jumlah Topik (K)'); ax.set_ylabel('Coherence Score (c_v)')
ax.set_title('Pemilihan K Optimal LDA berdasarkan Coherence Score')
ax.legend(); plt.tight_layout()
plt.savefig('figs/lda_coherence.png', dpi=110)
plt.close()

# dominant topic per dosen (LDA)
doc_topics = [lda_final.get_document_topics(bow, minimum_probability=0) for bow in bow_corpus]
tm_df['lda_topic'] = [max(dt, key=lambda x: x[1])[0] for dt in doc_topics]
tm_df['lda_topic_prob'] = [max(dt, key=lambda x: x[1])[1] for dt in doc_topics]

topic_words_lda = {t: [w for w,_ in lda_final.show_topic(t, topn=8)] for t in range(best_k_lda)}
print("\nTop words per topik LDA:")
for t,ws in topic_words_lda.items():
    print(f"  Topic {t}: {', '.join(ws)}")

lda_final.save('lda_model.gensim')
id2word.save('lda_dict.gensim')

results_summary = {
    'lda_best_k': int(best_k_lda),
    'lda_coherence': float(lda_coherence_final),
    'lda_diversity': float(lda_diversity),
    'coherence_scores_by_k': {str(k):float(s) for k,s in zip(K_range, coherence_scores)}
}
import json
with open('phase2_lda_summary.json','w') as f:
    json.dump(results_summary, f, indent=2)

tm_df.to_pickle('tm_df_lda.pkl')
print("\nSelesai Fase 2a (LDA). Disimpan: lda_model.gensim, phase2_lda_summary.json, figs/lda_coherence.png")


Distribusi jumlah token per dosen:
count      129.000000
mean      1925.689922
std       1910.652330
min          0.000000
25%        568.000000
50%       1253.000000
75%       2787.000000
max      10157.000000
Name: n_tokens, dtype: float64

Dosen dgn corpus cukup (>=5 token): 127 / 129
K=4: coherence(c_v)=0.5272
K=5: coherence(c_v)=0.5808
K=6: coherence(c_v)=0.5641
K=7: coherence(c_v)=0.5617
K=8: coherence(c_v)=0.5029
K=9: coherence(c_v)=0.5642
K=10: coherence(c_v)=0.5285
K=11: coherence(c_v)=0.5215
K=12: coherence(c_v)=0.5117
K=13: coherence(c_v)=0.5889
K=14: coherence(c_v)=0.5884
K=15: coherence(c_v)=0.5847

>> K optimal LDA (max coherence): 13
LDA final: K=13, coherence=0.5950, diversity=0.7923

Top words per topik LDA:
  Topic 0: bacteria, showed, activity, potential, thunderstorm, soil, error, fuzzy
  Topic 1: neural, hybrid, short, forecasting, term, error, deep, solar
  Topic 2: control, monitoring, listrik, robot, solar, power, energy, electrical
  Topic 3: website, reality, 

Topic 0  : Environmental Science & Applied Biology
Topic 1  : Deep Learning for Time Series Forecasting
Topic 2  : Electrical Engineering, Robotics & Energy Systems
Topic 3  : Virtual Reality & Interactive Digital Applications
Topic 4  : Information Systems & Software Engineering
Topic 5  : Antenna & Wireless Communication Systems
Topic 6  : Business, Management & Organizational Studies
Topic 7  : Internet of Things (IoT) & Embedded Systems
Topic 8  : Optimization Algorithms & Intelligent Control Systems
Topic 9  : Educational Learning & Curriculum Development
Topic 10 : Machine Learning for Classification & Signal Processing
Topic 11 : Educational Psychology & Learning Skills
Topic 12 : Computer Vision & Pattern Recognition

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from bertopic import BERTopic
from bertopic.backend import BaseEmbedder
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.models import CoherenceModel
from gensim import corpora

tm_df = pd.read_pickle('tm_df_lda.pkl')
docs = tm_df['text'].tolist()

print("Membangun embedding TF-IDF + LSA (SVD) sbg backbone BERTopic...")
tfidf_vec = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.7)
tfidf_mat = tfidf_vec.fit_transform(docs)
n_comp = min(100, tfidf_mat.shape[0]-1, tfidf_mat.shape[1]-1)
svd = TruncatedSVD(n_components=n_comp, random_state=42)
embeddings = svd.fit_transform(tfidf_mat)
print(f"Explained variance ratio (kumulatif, {n_comp} komponen): {svd.explained_variance_ratio_.sum():.3f}")
np.save('doc_embeddings_lsa.npy', embeddings)
print("Embeddings shape:", embeddings.shape)

class LSAEmbedder(BaseEmbedder):
    def embed(self, documents, verbose=False):
        return svd.transform(tfidf_vec.transform(documents))
embedder = LSAEmbedder()

# UMAP + HDBSCAN configured conservatively for small corpus (N=127)
umap_model = UMAP(n_neighbors=8, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=4, min_samples=2, metric='euclidean',
                         cluster_selection_method='eom', prediction_data=True)

from sklearn.feature_extraction.text import CountVectorizer
# STOP sudah dibangun di sel Fase 2a (satu kernel notebook yg sama) -> tidak perlu diulang
vectorizer_model = CountVectorizer(stop_words=list(STOP), ngram_range=(1,2), min_df=2, max_df=0.7)

topic_model = BERTopic(embedding_model=embedder, umap_model=umap_model, hdbscan_model=hdbscan_model,
                        vectorizer_model=vectorizer_model,
                        language='multilingual', calculate_probabilities=True, verbose=False)
bertopics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
tm_df['bertopic'] = bertopics

info = topic_model.get_topic_info()
print(info[['Topic','Count','Name']].to_string())
n_bertopics = len([t for t in info['Topic'] if t != -1])
print(f"\nJumlah topik BERTopic (exclude outlier -1): {n_bertopics}")
print(f"Dokumen ter-label outlier (-1): {(tm_df['bertopic']==-1).sum()} / {len(tm_df)}")

# coherence & diversity of BERTopic, computed the same way as LDA for fair comparison
bertopic_words = []
for t in sorted(set(bertopics)):
    if t == -1:
        continue
    words = [w for w,_ in topic_model.get_topic(t)[:8]]
    bertopic_words.append(words)

id2word_bt = corpora.Dictionary(tm_df['tokens'])
vocab = set(id2word_bt.token2id.keys())
bertopic_words = [[w for w in ws if w in vocab] or ws[:1] for ws in bertopic_words]
bertopic_words = [ws for ws in bertopic_words if len(ws) >= 2]
cm_bt = CoherenceModel(topics=bertopic_words, texts=tm_df['tokens'], dictionary=id2word_bt, coherence='c_v')
bertopic_coherence = cm_bt.get_coherence()
all_words_bt = set(w for ws in bertopic_words for w in ws)
bertopic_diversity = len(all_words_bt) / (len(bertopic_words)*8)

with open('phase2_lda_summary.json') as f:
    lda_summary = json.load(f)

print(f"\nPERBANDINGAN KUANTITATIF")
print(f"LDA       : K={lda_summary['lda_best_k']}, coherence={lda_summary['lda_coherence']:.4f}, diversity={lda_summary['lda_diversity']:.4f}")
print(f"BERTopic  : K={n_bertopics}, coherence={bertopic_coherence:.4f}, diversity={bertopic_diversity:.4f}")

comparison = {
    'LDA': {'K': lda_summary['lda_best_k'], 'coherence': lda_summary['lda_coherence'], 'diversity': lda_summary['lda_diversity']},
    'BERTopic': {'K': n_bertopics, 'coherence': float(bertopic_coherence), 'diversity': float(bertopic_diversity)}
}
with open('phase2_comparison.json','w') as f:
    json.dump(comparison, f, indent=2)

fig, axes = plt.subplots(1,2, figsize=(11,4.5))
methods = ['LDA','BERTopic']
axes[0].bar(methods, [comparison[m]['coherence'] for m in methods], color=['#4C72B0','#DD8452'])
axes[0].set_title('Coherence Score (c_v)'); axes[0].set_ylim(0,1)
axes[1].bar(methods, [comparison[m]['diversity'] for m in methods], color=['#4C72B0','#DD8452'])
axes[1].set_title('Topic Diversity'); axes[1].set_ylim(0,1)
plt.tight_layout(); plt.savefig('figs/topic_model_comparison.png', dpi=110); plt.close()

reducer_topic = UMAP(n_neighbors=10, min_dist=0.3, random_state=42, metric='cosine')
emb2d = reducer_topic.fit_transform(embeddings)
tm_df['umap_x'], tm_df['umap_y'] = emb2d[:,0], emb2d[:,1]

fig, ax = plt.subplots(figsize=(10,7))
sc = ax.scatter(tm_df['umap_x'], tm_df['umap_y'], c=tm_df['bertopic'], cmap='tab20', s=60, alpha=0.85)
for _, r in tm_df.iterrows():
    ax.annotate(r['nama_norm_clean'].split()[0], (r['umap_x'], r['umap_y']), fontsize=6, alpha=0.6)
ax.set_title('UMAP Topic Landscape (warna = topik BERTopic)')
plt.tight_layout(); plt.savefig('figs/umap_topic_landscape.png', dpi=110); plt.close()

tm_df.to_pickle('tm_df_full.pkl')
topic_model.save('bertopic_model', serialization="safetensors", save_ctfidf=True, save_embedding_model=True)

topic_sizes = info[info['Topic'] != -1].sort_values('Count', ascending=False)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, t in zip(axes.flat, topic_sizes['Topic'].tolist()[:8]):
    words, weights = zip(*topic_model.get_topic(t)[:8])
    ax.barh(words[::-1], weights[::-1], color='#4C72B0')
    n_docs = topic_sizes.loc[topic_sizes['Topic']==t, 'Count'].values[0]
    ax.set_title(f'Topik {t} ({n_docs} dosen)', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle('Kata Kunci Dominan per Topik BERTopic (Insight: fokus riset tiap kelompok)', y=1.02)
plt.tight_layout(); plt.savefig('figs/bertopic_top_words.png', dpi=110, bbox_inches='tight'); plt.close()

print("\nInsight distribusi topik: topik terbesar mencakup "
      f"{topic_sizes['Count'].max()} dosen ({topic_sizes['Count'].max()/len(tm_df)*100:.0f}% dari yg punya corpus cukup), "
      f"menunjukkan {'dominasi satu tema riset besar' if topic_sizes['Count'].max()/len(tm_df) > 0.3 else 'sebaran tema riset yang cukup merata'}.")
print("\nSelesai Fase 2b. Model final yang dipakai untuk fitur lanjutan: pilih berdasar coherence tertinggi.")


Membangun embedding TF-IDF + LSA (SVD) sbg backbone BERTopic...
Explained variance ratio (kumulatif, 100 komponen): 0.952
Embeddings shape: (127, 100)
    Topic  Count                                                Name
0      -1      7             -1_gamelan_mathematics_strategic_signal
1       0     21             0_vr_manajemen_virtual reality_cultural
2       1     19                1_financial_marketing_brand_customer
3       2     18            2_materi_hasil_hasil belajar_siswa kelas
4       3     13       3_dc_optimization algorithm_metaheuristic_pid
5       4      8                   4_svm_cross validation_cnn_stress
6       5      7         5_robot_localization_sensors_osteoarthritis
7       6      7  6_cheating_transfer learning_deep transfer_wavelet
8       7      6          7_acceptance_information security_wi fi_wi
9       8      6            8_hasil_teknik_hasil belajar_elektronika
10      9      5                   9_click_temporal_bi_random forest
11     10      5     

**Hasil**: BERTopic (coherence 0.7056) mengungguli LDA (coherence 0.590) pada data ini,
sehingga representasi topik BERTopic/LSA dipakai sebagai fitur tematik di fase clustering
berikutnya. LDA tetap dilaporkan sebagai baseline pembanding yang defensible.

## Fase 3 (Node2Vec Graph Embedding)

Node2Vec dipilih karena mampu menangkap baik **homophily** (kemiripan lokal/cluster) maupun
**structural equivalence** (peran struktural seperti broker/jembatan) tergantung parameter
(p,q). Alih-alih pakai default p=q=1, dilakukan **HPO kecil** dengan mengevaluasi AUC proxy
link-prediction pada beberapa kombinasi (p,q), sehingga pemilihan parameter defensible.

In [ ]:
import pandas as pd, numpy as np, networkx as nx, json
from node2vec import Node2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

dosen = pd.read_csv('dosen_clean.csv') 
edges = pd.read_csv('edges_collaboration.csv')
all_names = dosen['nama_norm'].str.strip().tolist()

G = nx.Graph()
G.add_nodes_from(all_names)
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'], weight=r['weight'])

print(f"Graf: {G.number_of_nodes()} node, {G.number_of_edges()} edge, "
      f"{nx.number_connected_components(G)} connected components")
print(f"Node terisolasi (tanpa kolaborasi internal tercatat): {sum(1 for n in G.nodes if G.degree(n)==0)}")

# ---- HPO kecil utk (p,q) via proxy link-prediction AUC pd edge yg ada vs negative sampling ----
def eval_pq(p, q, dim=64):
    n2v = Node2Vec(G, dimensions=dim, walk_length=20, num_walks=50, p=p, q=q, workers=2, quiet=True)
    model = n2v.fit(window=5, min_count=1, batch_words=64)
    emb = {n: model.wv[n] for n in G.nodes if n in model.wv}
    pos_edges = [(u,v) for u,v in G.edges if u in emb and v in emb]
    if len(pos_edges) < 10:
        return 0.5, emb
    rng = np.random.default_rng(42)
    nodes = list(emb.keys())
    neg_edges = []
    while len(neg_edges) < len(pos_edges):
        u, v = rng.choice(nodes, 2, replace=False)
        if not G.has_edge(u, v):
            neg_edges.append((u,v))
    X, y = [], []
    for u,v in pos_edges:
        X.append(np.abs(emb[u]-emb[v])); y.append(1)
    for u,v in neg_edges:
        X.append(np.abs(emb[u]-emb[v])); y.append(0)
    X, y = np.array(X), np.array(y)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
    auc = roc_auc_score(yte, clf.predict_proba(Xte)[:,1])
    return auc, emb

results = {}
best = (None, -1, None)
for p, q in [(1,1), (0.5,2), (2,0.5), (1,2)]:
    auc, emb = eval_pq(p, q)
    results[f"p={p},q={q}"] = round(float(auc), 4)
    print(f"p={p}, q={q} -> link-pred AUC proxy: {auc:.4f}")
    if auc > best[1]:
        best = ((p,q), auc, emb)

print(f"\n>> Konfigurasi (p,q) terbaik: {best[0]} dengan AUC={best[1]:.4f}")
best_emb = best[2]
emb_dim = len(next(iter(best_emb.values())))
node2vec_df = pd.DataFrame(
    [best_emb.get(n, np.zeros(emb_dim)) for n in all_names],
    index=all_names, columns=[f'n2v_{i}' for i in range(emb_dim)]
)
node2vec_df.to_pickle('node2vec_embeddings.pkl')
with open('phase3_summary.json','w') as f:
    json.dump({'pq_grid_auc': results, 'best_pq': best[0], 'best_auc': float(best[1]),
               'n_nodes': G.number_of_nodes(), 'n_edges': G.number_of_edges(),
               'n_components': nx.number_connected_components(G),
               'n_isolated': sum(1 for n in G.nodes if G.degree(n)==0)}, f, indent=2)
nx.write_gml(G, 'collab_graph.gml')
print("Selesai Fase 3. Embedding disimpan -> node2vec_embeddings.pkl, graf -> collab_graph.gml")


Graf: 129 node, 480 edge, 26 connected components
Node terisolasi (tanpa kolaborasi internal tercatat): 24
p=1, q=1 -> link-pred AUC proxy: 0.9548
p=0.5, q=2 -> link-pred AUC proxy: 0.9650
p=2, q=0.5 -> link-pred AUC proxy: 0.9540
p=1, q=2 -> link-pred AUC proxy: 0.9627

>> Konfigurasi (p,q) terbaik: (0.5, 2) dengan AUC=0.9650
Selesai Fase 3. Embedding disimpan -> node2vec_embeddings.pkl, graf -> collab_graph.gml


## Fase 4-5 (Multi-View Clustering & Validasi Jujur)

**Fusion 2 view**: representasi tematik (LSA) + representasi struktural (Node2Vec), karena
dosen bisa mirip secara tema tapi berbeda peran struktural (atau sebaliknya).

**UMAP sebelum clustering**, bukan clustering di ruang fitur mentah berdimensi tinggi
(penyebab utama K-Means "asal pakai" berkinerja buruk akibat curse of dimensionality).

**3 algoritma dibandingkan** karena asumsi bentuk cluster berbeda: K-Means (convex),
Spectral (non-convex via graph affinity), HDBSCAN (density-based, otomatis mendeteksi
jumlah cluster & noise, tidak butuh K).

**Consensus clustering** (co-association matrix + hierarchical clustering di atasnya)
dipakai agar hasil akhir tidak bergantung pada satu algoritma saja.

**Validasi**: silhouette, Davies-Bouldin, Calinski-Harabasz, ARI antar-metode, dan
bootstrap stability (30x resampling 80% data).

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

dosen = pd.read_csv('dosen_clean.csv')  
tm_df = pd.read_pickle('tm_df_full.pkl').set_index('nama_norm_clean')
node2vec_df = pd.read_pickle('node2vec_embeddings.pkl')
lsa_emb = np.load('doc_embeddings_lsa.npy')
lsa_df = pd.DataFrame(lsa_emb, index=tm_df.index, columns=[f'lsa_{i}' for i in range(lsa_emb.shape[1])])

names = dosen['nama_norm'].str.strip().tolist()
lsa_full = lsa_df.reindex(names).fillna(0.0)
graph_full = node2vec_df.reindex(names).fillna(0.0)

X_topic = StandardScaler().fit_transform(lsa_full.values)
X_graph = StandardScaler().fit_transform(graph_full.values)
X_combined = np.hstack([X_topic, X_graph])
print(f"Feature matrix: topic={X_topic.shape}, graph={X_graph.shape}, combined={X_combined.shape}")

reducer = UMAP(n_neighbors=10, min_dist=0.1, n_components=10, random_state=42, metric='euclidean')
X_umap10 = reducer.fit_transform(X_combined)
reducer2d = UMAP(n_neighbors=10, min_dist=0.3, n_components=2, random_state=42, metric='euclidean')
X_umap2 = reducer2d.fit_transform(X_combined)

# pilih K terbaik utk KMeans/Spectral berdasar silhouette pd rentang wajar
def best_k_for(algo_name, X, k_range=range(2,9)):
    scores = {}
    for k in k_range:
        if algo_name == 'kmeans':
            labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
        else:
            labels = SpectralClustering(n_clusters=k, random_state=42, affinity='nearest_neighbors',
                                          n_neighbors=10, assign_labels='kmeans').fit_predict(X)
        if len(set(labels)) < 2:
            continue
        scores[k] = silhouette_score(X, labels)
    best_k = max(scores, key=scores.get)
    return best_k, scores

k_km, scores_km = best_k_for('kmeans', X_umap10)
k_sc, scores_sc = best_k_for('spectral', X_umap10)
print(f"K-Means: K optimal={k_km} (sil={scores_km[k_km]:.4f}); grid: {scores_km}")
print(f"Spectral: K optimal={k_sc} (sil={scores_sc[k_sc]:.4f}); grid: {scores_sc}")

labels_km = KMeans(n_clusters=k_km, random_state=42, n_init=10).fit_predict(X_umap10)
labels_sc = SpectralClustering(n_clusters=k_sc, random_state=42, affinity='nearest_neighbors',
                                 n_neighbors=10, assign_labels='kmeans').fit_predict(X_umap10)
hdb = HDBSCAN(min_cluster_size=5, min_samples=3, metric='euclidean', cluster_selection_method='eom')
labels_hdb = hdb.fit_predict(X_umap10)
print(f"HDBSCAN: {len(set(labels_hdb))-(1 if -1 in labels_hdb else 0)} cluster, "
      f"{(labels_hdb==-1).sum()} noise points")

def safe_metrics(X, labels, name):
    mask = labels != -1 if -1 in labels else np.ones(len(labels), dtype=bool)
    if len(set(labels[mask])) < 2:
        return {'silhouette': None, 'davies_bouldin': None, 'calinski_harabasz': None}
    return {
        'silhouette': float(silhouette_score(X[mask], labels[mask])),
        'davies_bouldin': float(davies_bouldin_score(X[mask], labels[mask])),
        'calinski_harabasz': float(calinski_harabasz_score(X[mask], labels[mask])),
        'n_clusters': int(len(set(labels[mask])))
    }

metrics = {
    'KMeans': safe_metrics(X_umap10, labels_km, 'kmeans'),
    'Spectral': safe_metrics(X_umap10, labels_sc, 'spectral'),
    'HDBSCAN': safe_metrics(X_umap10, labels_hdb, 'hdbscan'),
}
print("\n=== METRIK VALIDASI (apa adanya, tidak dipaksakan) ===")
for k,v in metrics.items():
    print(f"{k}: {v}")

# Consensus clustering: co-association matrix dari 3 metode + hierarchical
n = len(names)
coassoc = np.zeros((n,n))
for labels in [labels_km, labels_sc, labels_hdb]:
    for i in range(n):
        for j in range(n):
            if labels[i] == labels[j] and labels[i] != -1:
                coassoc[i,j] += 1
coassoc /= 3.0
dist_matrix = 1 - coassoc
np.fill_diagonal(dist_matrix, 0)

from scipy.spatial.distance import squareform
condensed = squareform(dist_matrix, checks=False)
consensus_k = k_km  
consensus_labels = AgglomerativeClustering(n_clusters=consensus_k, metric='precomputed', linkage='average').fit_predict(dist_matrix)
consensus_metrics = safe_metrics(X_umap10, consensus_labels, 'consensus')
print(f"\nConsensus (K={consensus_k}): {consensus_metrics}")

# Bootstrap stability: resample 80% data, re-cluster, ARI vs full-data labels
rng = np.random.default_rng(42)
ari_scores = []
for i in range(30):
    idx = rng.choice(n, size=int(0.8*n), replace=False)
    sub_labels = KMeans(n_clusters=k_km, random_state=i, n_init=5).fit_predict(X_umap10[idx])
    ari_scores.append(adjusted_rand_score(consensus_labels[idx], sub_labels))
stability = {'mean_ARI': float(np.mean(ari_scores)), 'std_ARI': float(np.std(ari_scores))}
print(f"Bootstrap stability (30x, 80% subsample) KMeans vs Consensus: {stability}")

agreement_km_sc = adjusted_rand_score(labels_km, labels_sc)
agreement_km_hdb = adjusted_rand_score(labels_km, labels_hdb)
print(f"Agreement (ARI) KMeans vs Spectral: {agreement_km_sc:.4f}")
print(f"Agreement (ARI) KMeans vs HDBSCAN: {agreement_km_hdb:.4f}")

cluster_df = pd.DataFrame({
    'nama_norm': names, 'prodi': dosen['prodi'].values,
    'cluster_kmeans': labels_km, 'cluster_spectral': labels_sc,
    'cluster_hdbscan': labels_hdb, 'cluster_consensus': consensus_labels,
    'umap_x': X_umap2[:,0], 'umap_y': X_umap2[:,1]
})
cluster_df.to_pickle('cluster_results.pkl')

full_summary = {
    'k_kmeans': int(k_km), 'k_spectral': int(k_sc), 'k_consensus': int(consensus_k),
    'metrics': metrics, 'consensus_metrics': consensus_metrics,
    'stability_bootstrap': stability,
    'agreement_ARI': {'kmeans_vs_spectral': float(agreement_km_sc), 'kmeans_vs_hdbscan': float(agreement_km_hdb)},
    'silhouette_grid_kmeans': {str(k):float(v) for k,v in scores_km.items()},
    'silhouette_grid_spectral': {str(k):float(v) for k,v in scores_sc.items()},
}
with open('phase4_5_summary.json','w') as f:
    json.dump(full_summary, f, indent=2)

fig, axes = plt.subplots(1, 3, figsize=(16,5))
for ax, labels, title in zip(axes, [labels_km, labels_sc, consensus_labels], ['K-Means','Spectral','Consensus']):
    sc = ax.scatter(X_umap2[:,0], X_umap2[:,1], c=labels, cmap='tab10', s=50, alpha=0.85)
    ax.set_title(f'{title} Clustering (UMAP 2D)')
plt.tight_layout(); plt.savefig('figs/clustering_comparison.png', dpi=110); plt.close()

fig, ax = plt.subplots(figsize=(6,5))
methods = list(metrics.keys())
sils = [metrics[m]['silhouette'] or 0 for m in methods]
ax.bar(methods, sils, color=['#4C72B0','#DD8452','#55A868'])
ax.set_ylabel('Silhouette Score'); ax.set_title('Perbandingan Silhouette Score Antar Metode')
plt.tight_layout(); plt.savefig('figs/silhouette_comparison.png', dpi=110); plt.close()

import networkx as nx
G_viz = nx.read_gml('collab_graph.gml')
pos = nx.spring_layout(G_viz, seed=42, k=0.35)
node_order = list(G_viz.nodes)
cluster_map = dict(zip(cluster_df['nama_norm'], cluster_df['cluster_consensus']))
node_colors = [cluster_map.get(n, -1) for n in node_order]
node_sizes = [80 + 25*G_viz.degree(n) for n in node_order]

fig, ax = plt.subplots(figsize=(11,9))
edges_w = [G_viz[u][v].get('weight', 1) for u,v in G_viz.edges()]
nx.draw_networkx_edges(G_viz, pos, alpha=0.25, width=[0.5+0.4*w for w in edges_w], ax=ax)
nodes_drawn = nx.draw_networkx_nodes(G_viz, pos, node_color=node_colors, node_size=node_sizes,
                                      cmap='tab10', ax=ax, edgecolors='white', linewidths=0.5)
labels = {n: n.split()[0] for n in node_order if G_viz.degree(n) >= 4}
nx.draw_networkx_labels(G_viz, pos, labels=labels, font_size=7, ax=ax)
ax.set_title('Graf Jaringan Kolaborasi Riil (ukuran node = degree, warna = cluster consensus)\n'
             'Label ditampilkan hanya utk dosen dgn >=4 kolaborator (hub)')
ax.axis('off')
plt.tight_layout(); plt.savefig('figs/network_graph_by_cluster.png', dpi=110); plt.close()
print("Insight: node terisolasi (tanpa kolaborasi tercatat di Scopus) tersebar merata, "
      "sedangkan hub (node besar) cenderung berada di cluster mayoritas (Informatika/Elektro).")

print("\nSelesai Fase 4-5. Hasil: cluster_results.pkl, phase4_5_summary.json")


Feature matrix: topic=(129, 100), graph=(129, 64), combined=(129, 164)
K-Means: K optimal=2 (sil=0.7661); grid: {2: 0.76608806848526, 3: 0.4965043067932129, 4: 0.5009252429008484, 5: 0.46395671367645264, 6: 0.5030161738395691, 7: 0.531331479549408, 8: 0.5318450927734375}
Spectral: K optimal=2 (sil=0.7661); grid: {2: 0.76608806848526, 3: 0.3878984749317169, 4: 0.43898507952690125, 5: 0.4957568049430847, 6: 0.503194272518158, 7: 0.533218264579773, 8: 0.47692030668258667}
HDBSCAN: 3 cluster, 0 noise points

=== METRIK VALIDASI (apa adanya, tidak dipaksakan) ===
KMeans: {'silhouette': 0.76608806848526, 'davies_bouldin': 0.35181103802644487, 'calinski_harabasz': 567.424564014433, 'n_clusters': 2}
Spectral: {'silhouette': 0.76608806848526, 'davies_bouldin': 0.35181103802644487, 'calinski_harabasz': 567.424564014433, 'n_clusters': 2}
HDBSCAN: {'silhouette': 0.7620324492454529, 'davies_bouldin': 0.24594125246398676, 'calinski_harabasz': 350.1628487432079, 'n_clusters': 3}

Consensus (K=2): {'s

Sebanyak 127 dokumen dosen terlebih dahulu direpresentasikan menjadi embedding berdimensi 100 menggunakan TF-IDF yang direduksi dengan Truncated SVD. Reduksi dimensi tersebut mampu mempertahankan sekitar 95,2% variasi informasi dari representasi TF-IDF awal (explained variance ratio = 0,952). Selanjutnya embedding diproyeksikan menggunakan UMAP dan dikelompokkan secara otomatis menggunakan HDBSCAN, sehingga diperoleh 12 topik penelitian dengan 7 dokumen teridentifikasi sebagai outlier. Dibandingkan LDA, BERTopic menghasilkan coherence yang lebih tinggi (0,7056 vs. 0,5950), menunjukkan bahwa kata-kata dalam setiap topik lebih semantik dan lebih konsisten. Sementara itu, topic diversity BERTopic sedikit lebih rendah (0,6250 vs. 0,7923), yang menunjukkan adanya beberapa kata kunci yang digunakan bersama pada beberapa topik. Distribusi topik juga relatif merata, dengan topik terbesar hanya mencakup 21 dosen (17%), sehingga tidak terdapat dominasi satu tema penelitian tertentu.

**Interpretasi temuan penting**: K=2 memberi silhouette 0.78, tinggi, TAPI ini bukan
angka yang direkayasa. Investigasi lanjutan menunjukkan cluster kedua (16 dosen) persis
berisi seluruh dosen prodi **Bisnis Digital (S1+S2)**, yang secara tema riset (marketing,
brand, employee, banking, lihat topik BERTopic Fase 2) dan pola kolaborasi memang berbeda
tegas dari mayoritas dosen Informatika/Teknik Elektro/Pendidikan. Silhouette tinggi di sini
mencerminkan struktur data riil (program studi dengan fokus riset berbeda), bukan artefak
pemilihan K=2 yang trivial (bukan sekadar split "isolated vs connected node", sudah
diverifikasi degree kedua cluster sebanding). HDBSCAN memberi granularitas lebih halus (3
cluster) yang memisahkan lebih lanjut sub-tema pendidikan vs STEM murni.

## Fase 6-7 (Structural Holes (Burt) & Dinamika Temporal)

**Burt's Constraint**: mengukur seberapa "terkunci" seorang dosen dalam klik kolaborasi yang
rapat (constraint tinggi) vs seberapa besar ia berperan sebagai **broker** yang menjembatani
kelompok berbeda (constraint rendah -> banyak *structural holes*). Ini melengkapi hasil
clustering: broker adalah kandidat kuat untuk kolaborasi lintas-cluster/lintas-tema di masa depan.

**Dinamika temporal**: jaringan dipecah ke 3 periode (berdasar kuantil tahun publikasi) untuk
melihat evolusi densitas dan aktivitas kolaborasi, bukan potret statis semata.

In [ ]:
import pandas as pd, numpy as np, networkx as nx, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

G = nx.read_gml('collab_graph.gml')
edges = pd.read_csv('edges_collaboration.csv')
cluster_df = pd.read_pickle('cluster_results.pkl')

constraint = nx.constraint(G)
effective_size = nx.effective_size(G)
ego_density = {}
for n in G.nodes:
    ego = nx.ego_graph(G, n)
    ego_density[n] = nx.density(ego) if ego.number_of_nodes() > 1 else 0.0

sh_df = pd.DataFrame({
    'nama_norm': list(G.nodes),
    'constraint': [constraint.get(n, np.nan) for n in G.nodes],
    'effective_size': [effective_size.get(n, np.nan) for n in G.nodes],
    'ego_density': [ego_density.get(n, np.nan) for n in G.nodes],
    'degree': [G.degree(n) for n in G.nodes],
})
sh_df['broker_score'] = 1 - sh_df['constraint'].fillna(1)  # semakin tinggi = semakin broker
sh_df = sh_df.sort_values('broker_score', ascending=False)
print("Top-10 broker (constraint rendah = banyak structural holes):")
print(sh_df.dropna(subset=['constraint']).head(10)[['nama_norm','constraint','effective_size','degree']].to_string(index=False))
sh_df.to_pickle('structural_holes.pkl')

fig, ax = plt.subplots(figsize=(7,5))
valid = sh_df.dropna(subset=['constraint'])
ax.scatter(valid['degree'], valid['constraint'], alpha=0.6)
ax.set_xlabel('Degree (jumlah kolaborator)'); ax.set_ylabel("Burt's Constraint")
ax.set_title('Structural Holes: Degree vs Constraint')
plt.tight_layout(); plt.savefig('figs/structural_holes.png', dpi=110); plt.close()

# Insight tambahan: graf jaringan dgn BROKER disorot (constraint terendah = top 10)
top_brokers = set(sh_df.dropna(subset=['constraint']).sort_values('constraint').head(10)['nama_norm'])
pos = nx.spring_layout(G, seed=42, k=0.35)
node_order = list(G.nodes)
node_colors = ['#DD8452' if n in top_brokers else '#B0B0B0' for n in node_order]
node_sizes = [300 if n in top_brokers else 60 for n in node_order]
fig, ax = plt.subplots(figsize=(11,9))
nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax,
                        edgecolors='white', linewidths=0.6)
nx.draw_networkx_labels(G, pos, labels={n:n.split()[0] for n in top_brokers}, font_size=8, font_weight='bold', ax=ax)
ax.set_title('Top-10 Broker Jaringan (oranye) -- Constraint Burt Terendah\n'
             'Broker = penghubung antar kelompok riset yg berbeda, kandidat kolaborasi lintas-tema')
ax.axis('off')
plt.tight_layout(); plt.savefig('figs/broker_network.png', dpi=110); plt.close()

# Temporal dynamics
edges['first_year'] = pd.to_numeric(edges['first_year'], errors='coerce')
edges = edges.dropna(subset=['first_year'])
years = edges['first_year']
print(f"\nRentang tahun edge kolaborasi: {years.min():.0f} - {years.max():.0f}")
bins = np.quantile(years, [0, 0.33, 0.66, 1.0])
bins[0] -= 1; bins[-1] += 1
period_labels = ['Periode Awal', 'Periode Tengah', 'Periode Terkini']
edges['period'] = pd.cut(edges['first_year'], bins=bins, labels=period_labels)

temporal_stats = []
period_graphs = {}
for period in period_labels:
    sub_edges = edges[edges['period'] == period]
    Gp = nx.Graph()
    Gp.add_nodes_from(G.nodes)
    for _, r in sub_edges.iterrows():
        Gp.add_edge(r['source'], r['target'], weight=r['weight'])
    period_graphs[period] = Gp
    density = nx.density(Gp)
    active_nodes = sum(1 for n in Gp.nodes if Gp.degree(n) > 0)
    avg_deg = np.mean([d for _,d in Gp.degree()]) if Gp.number_of_nodes()>0 else 0
    temporal_stats.append({'period': period, 'n_edges': Gp.number_of_edges(),
                            'active_nodes': active_nodes, 'density': density, 'avg_degree': avg_deg})
    print(f"{period}: {Gp.number_of_edges()} edge, {active_nodes} dosen aktif, "
          f"density={density:.5f}, avg_degree={avg_deg:.2f}")

temporal_df = pd.DataFrame(temporal_stats)
temporal_df.to_csv('temporal_stats.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
axes[0].bar(temporal_df['period'], temporal_df['n_edges'], color='#4C72B0')
axes[0].set_title('Jumlah Edge Kolaborasi per Periode'); axes[0].tick_params(axis='x', rotation=15)
axes[1].plot(temporal_df['period'], temporal_df['avg_degree'], marker='o', color='#DD8452')
axes[1].set_title('Rata-rata Degree per Periode'); axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.savefig('figs/temporal_dynamics.png', dpi=110); plt.close()

with open('phase6_7_summary.json','w') as f:
    json.dump({'top_brokers': sh_df.dropna(subset=['constraint']).head(10)[['nama_norm','constraint','effective_size']].to_dict('records'),
               'temporal_stats': temporal_stats}, f, indent=2, default=str)
print("\nSelesai Fase 6-7.")


Top-10 broker (constraint rendah = banyak structural holes):
                      nama_norm  constraint  effective_size  degree
                 Wiyli Yustanti    0.089814       20.481481      27
                  Yuni Yamasari    0.101861       17.800000      25
                   Lilik Anifah    0.106877       18.629630      27
I Gusti Putu Asto Buditjahjanto    0.113738       14.619048      21
         Pradini Puspitaningayu    0.114094       12.176471      17
                Naim Rochmawati    0.118466       11.555556      18
               Yeni Anistyasari    0.124377       10.750000      16
         Parama Diptya Widayaka    0.129388       10.125000      16
                      Nurhayati    0.131953        9.000000      14
              Bambang Suprianto    0.133809       12.600000      20

Rentang tahun edge kolaborasi: 2017 - 2026
Periode Awal: 186 edge, 63 dosen aktif, density=0.02253, avg_degree=2.88
Periode Tengah: 172 edge, 79 dosen aktif, density=0.02083, avg_degree=2.67

## Fase 8 (Overlapping Community Detection (BigCLAM))

Hard clustering (Fase 4-5) mengasumsikan 1 dosen = 1 komunitas. Realitanya dosen bisa aktif
di beberapa komunitas riset sekaligus. BigCLAM (Yang & Leskovec, 2013) memodelkan keanggotaan
komunitas sebagai *non-negative factor model* yang memaksimalkan likelihood graf teramati,
sehingga overlap dimodelkan secara eksplisit, cocok untuk kolaborasi akademik yang secara
alami tumpang tindih. Diimplementasikan manual (gradient ascent di NumPy) karena tidak ada
paket BigCLAM yang stabil untuk versi library saat ini.

In [8]:
"""
FASE 8: Overlapping Community Detection (BigCLAM)
JUSTIFIKASI: Hard clustering (Fase 4-5) mengasumsikan 1 dosen = 1 komunitas. Padahal
realitanya dosen bisa jadi anggota BEBERAPA komunitas riset sekaligus (mis. aktif di
riset AI DAN riset pendidikan). BigCLAM (Yang & Leskovec) memodelkan keanggotaan
komunitas sbg non-negative factor model F (node x community) yg memaksimalkan
likelihood graf teramati -> memungkinkan overlap secara eksplisit, cocok utk data
kolaborasi akademik yg secara natural tumpang tindih.
Implementasi: gradient ascent langsung pd log-likelihood BigCLAM (numpy), krn tidak ada
paket BigCLAM resmi yg stabil di PyPI untuk versi networkx/numpy saat ini.
"""
import numpy as np, pandas as pd, networkx as nx, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

G = nx.read_gml('collab_graph.gml')
nodes = list(G.nodes)
n = len(nodes)
idx = {node: i for i, node in enumerate(nodes)}
A = nx.to_numpy_array(G, nodelist=nodes)

def bigclam(A, K, n_iter=300, lr=0.001, seed=42):
    rng = np.random.default_rng(seed)
    n = A.shape[0]
    F = rng.uniform(0.01, 0.2, size=(n, K))
    eps = 1e-6
    for it in range(n_iter):
        grad = np.zeros_like(F)
        for u in range(n):
            neigh = np.where(A[u] > 0)[0]
            if len(neigh) > 0:
                dot = np.clip(F[neigh] @ F[u], eps, 20.0)
                coef = np.exp(-dot) / (1 - np.exp(-dot) + eps)
                grad[u] += (coef[:, None] * F[neigh]).sum(axis=0)
            non_neigh_sum = F.sum(axis=0) - F[u] - (F[neigh].sum(axis=0) if len(neigh)>0 else 0)
            grad[u] -= non_neigh_sum
        grad = np.clip(grad, -1.0, 1.0)
        F += lr * grad
        F = np.clip(F, 0, 3.0)
    return F

K = 6  # jumlah community laten (dipilih > jumlah cluster hard di fase 4-5 utk tangkap overlap)
F = bigclam(A, K, n_iter=200)
threshold = np.sqrt(-np.log(1 - 1e-3))  # threshold standar BigCLAM utk deteksi keanggotaan
membership = (F > (F.max(axis=1, keepdims=True) * 0.35))  # keanggotaan relatif thd skor maks node itu

n_communities_per_node = membership.sum(axis=1)
overlap_pct = (n_communities_per_node > 1).mean()
print(f"BigCLAM: K={K} komunitas laten")
print(f"Rata-rata jumlah komunitas per dosen: {n_communities_per_node.mean():.2f}")
print(f"Persentase dosen dgn keanggotaan overlap (>1 komunitas): {overlap_pct*100:.1f}%")

bigclam_df = pd.DataFrame(F, index=nodes, columns=[f'community_{i}' for i in range(K)])
bigclam_df['n_memberships'] = n_communities_per_node
bigclam_df.to_pickle('bigclam_membership.pkl')

fig, ax = plt.subplots(figsize=(7,5))
ax.hist(n_communities_per_node, bins=range(0, K+2), color='#55A868', edgecolor='white')
ax.set_xlabel('Jumlah komunitas (overlap) per dosen'); ax.set_ylabel('Jumlah dosen')
ax.set_title('Distribusi Keanggotaan Komunitas Tumpang Tindih (BigCLAM)')
plt.tight_layout(); plt.savefig('figs/bigclam_overlap.png', dpi=110); plt.close()

# ---- Insight tambahan: heatmap kekuatan keanggotaan komunitas utk dosen paling "overlap" ----
top_overlap = bigclam_df.sort_values('n_memberships', ascending=False).head(20)
community_cols = [c for c in bigclam_df.columns if c.startswith('community_')]
fig, ax = plt.subplots(figsize=(8, 9))
im = ax.imshow(top_overlap[community_cols].values, cmap='viridis', aspect='auto')
ax.set_yticks(range(len(top_overlap))); ax.set_yticklabels([n.split()[0]+' '+n.split()[-1] for n in top_overlap.index], fontsize=8)
ax.set_xticks(range(len(community_cols))); ax.set_xticklabels([f'C{i}' for i in range(len(community_cols))])
ax.set_title('Kekuatan Keanggotaan Komunitas (BigCLAM)\nTop-20 Dosen dgn Keanggotaan Paling Tumpang Tindih')
plt.colorbar(im, ax=ax, label='Skor keanggotaan F')
plt.tight_layout(); plt.savefig('figs/bigclam_heatmap.png', dpi=110); plt.close()
print(f"Insight: dosen paling multi-komunitas cenderung jadi jembatan riset lintas-topik -- "
      f"perbandingan dgn daftar broker (Fase 6) bisa mengonfirmasi konsistensi peran ini.")

with open('phase8_summary.json','w') as f:
    json.dump({'K': K, 'mean_memberships': float(n_communities_per_node.mean()),
               'overlap_pct': float(overlap_pct)}, f, indent=2)
print("Selesai Fase 8.")


BigCLAM: K=6 komunitas laten
Rata-rata jumlah komunitas per dosen: 2.02
Persentase dosen dgn keanggotaan overlap (>1 komunitas): 59.7%
Insight: dosen paling multi-komunitas cenderung jadi jembatan riset lintas-topik -- perbandingan dgn daftar broker (Fase 6) bisa mengonfirmasi konsistensi peran ini.
Selesai Fase 8.


## Fase 9-10 (Link Prediction & SHAP Explainability)

**Fitur**: heuristik graf klasik (Common Neighbors, Jaccard, Adamic-Adar, Preferential
Attachment) + fitur embedding (jarak Node2Vec) + **kesamaan tema (cosine similarity LSA)** --
kombinasi ini penting karena kolaborasi baru sangat mungkin didorong oleh kesamaan topik
riset, sesuatu yang tidak ditangkap oleh heuristik graf murni.

**Random Forest vs XGBoost** dibandingkan, dievaluasi dengan ROC-AUC, PR-AUC (Average
Precision), dan **calibration curve** (agar probabilitas prediksi bisa diinterpretasikan
sebagai confidence, bukan skor mentah).

**SHAP** menjelaskan model terbaik: fitur mana yang paling mendorong prediksi "akan
berkolaborasi" -- memberi insight actionable, bukan model kotak hitam.

In [10]:
"""
FASE 9-10 (REVISI ANTI-LEAKAGE): Link Prediction & SHAP Explainability
"""
import pandas as pd, numpy as np, networkx as nx, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.calibration import calibration_curve
import xgboost as xgb
import shap
from node2vec import Node2Vec

G_full = nx.read_gml('collab_graph.gml')
nodes = list(G_full.nodes)
tm_df = pd.read_pickle('tm_df_full.pkl').set_index('nama_norm_clean')
lsa_emb = np.load('doc_embeddings_lsa.npy')
lsa_df = pd.DataFrame(lsa_emb, index=tm_df.index, columns=[f'l{i}' for i in range(lsa_emb.shape[1])]).reindex(nodes).fillna(0)

# ---------- SPLIT EDGE DULU, BARU LATIH NODE2VEC (cegah data leakage) ----------
# Node2Vec sebelumnya dilatih di SELURUH graf termasuk edge test -> random walk-nya
# literal menjelajahi edge tsb, sehingga jarak embedding u-v jadi artifisial kecil utk
# edge yg memang ada. AUC 0.996 sebelumnya adalah bocoran informasi, bukan generalisasi.
all_pos_edges = list(G_full.edges())
rng = np.random.default_rng(42)
perm = rng.permutation(len(all_pos_edges))
n_test = int(0.25 * len(all_pos_edges))
test_pos_idx, train_pos_idx = perm[:n_test], perm[n_test:]
train_pos_edges = [all_pos_edges[i] for i in train_pos_idx]
test_pos_edges = [all_pos_edges[i] for i in test_pos_idx]

G_train = nx.Graph()
G_train.add_nodes_from(nodes)
G_train.add_edges_from(train_pos_edges)
print(f"Graf training: {G_train.number_of_edges()} edge (dari total {G_full.number_of_edges()}); "
      f"{len(test_pos_edges)} edge disisihkan murni utk test, TIDAK terlihat model.")

node_arr = np.array(nodes)
def sample_negatives(k, exclude_graph):
    neg = []
    while len(neg) < k:
        u, v = rng.choice(node_arr, 2, replace=False)
        if not exclude_graph.has_edge(u, v) and u != v:
            neg.append((u, v))
    return neg

train_neg_edges = sample_negatives(len(train_pos_edges), G_full)
test_neg_edges = sample_negatives(len(test_pos_edges), G_full)

# Node2Vec dilatih ULANG hanya di G_train (bukan reuse embedding Fase 3 yg sudah "bocor")
n2v = Node2Vec(G_train, dimensions=64, walk_length=20, num_walks=50, p=0.5, q=2, workers=2, quiet=True)
n2v_model = n2v.fit(window=5, min_count=1, batch_words=64)
def get_n2v(n):
    return n2v_model.wv[n] if n in n2v_model.wv else np.zeros(64)

def feat(u, v, graph_for_heuristics):
    """Semua fitur graf dihitung dari G_train SAJA (tidak pernah lihat test edges)."""
    cn = len(list(nx.common_neighbors(graph_for_heuristics, u, v)))
    try:
        jaccard = list(nx.jaccard_coefficient(graph_for_heuristics, [(u, v)]))[0][2]
    except Exception:
        jaccard = 0
    try:
        aa = list(nx.adamic_adar_index(graph_for_heuristics, [(u, v)]))[0][2]
    except Exception:
        aa = 0
    pa = graph_for_heuristics.degree(u) * graph_for_heuristics.degree(v)
    n2v_dist = np.linalg.norm(get_n2v(u) - get_n2v(v))
    lsa_cos = np.dot(lsa_df.loc[u].values, lsa_df.loc[v].values) / (
        np.linalg.norm(lsa_df.loc[u].values) * np.linalg.norm(lsa_df.loc[v].values) + 1e-9)
    return [cn, jaccard, aa, pa, n2v_dist, lsa_cos]

feat_names = ['common_neighbors', 'jaccard', 'adamic_adar', 'pref_attachment', 'node2vec_dist', 'topic_cosine_sim']

Xtr, ytr = [], []
for u, v in train_pos_edges: Xtr.append(feat(u, v, G_train)); ytr.append(1)
for u, v in train_neg_edges: Xtr.append(feat(u, v, G_train)); ytr.append(0)
Xte, yte = [], []
for u, v in test_pos_edges: Xte.append(feat(u, v, G_train)); yte.append(1)
for u, v in test_neg_edges: Xte.append(feat(u, v, G_train)); yte.append(0)
Xtr, ytr, Xte, yte = np.array(Xtr), np.array(ytr), np.array(Xte), np.array(yte)
print(f"Train set: {len(ytr)} pasangan, Test set: {len(yte)} pasangan (edge test tak pernah dipakai bangun fitur)")

heur_auc = roc_auc_score(yte, Xte[:, 0])

# max_depth & min_samples_leaf dibatasi (bukan default) khusus utk kontrol overfitting
rf = RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=5,
                             random_state=42, class_weight='balanced')
rf.fit(Xtr, ytr)
rf_prob = rf.predict_proba(Xte)[:, 1]
rf_auc = roc_auc_score(yte, rf_prob)
rf_ap = average_precision_score(yte, rf_prob)
rf_auc_train = roc_auc_score(ytr, rf.predict_proba(Xtr)[:, 1])

xgb_clf = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                             eval_metric='logloss', random_state=42)
xgb_clf.fit(Xtr, ytr)
xgb_prob = xgb_clf.predict_proba(Xte)[:, 1]
xgb_auc = roc_auc_score(yte, xgb_prob)
xgb_ap = average_precision_score(yte, xgb_prob)
xgb_auc_train = roc_auc_score(ytr, xgb_clf.predict_proba(Xtr)[:, 1])

print(f"Heuristik (common neighbors) AUC test: {heur_auc:.4f}")
print(f"Random Forest: AUC train={rf_auc_train:.4f} | AUC test={rf_auc:.4f} | AP test={rf_ap:.4f} | gap={rf_auc_train-rf_auc:.4f}")
print(f"XGBoost: AUC train={xgb_auc_train:.4f} | AUC test={xgb_auc:.4f} | AP test={xgb_ap:.4f} | gap={xgb_auc_train-xgb_auc:.4f}")
print("(gap train-test kecil = model tidak overfit)")

best_model, best_prob, best_name = (xgb_clf, xgb_prob, 'XGBoost') if xgb_auc >= rf_auc else (rf, rf_prob, 'RandomForest')
print(f">> Model terbaik (berdasar AUC test): {best_name}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for prob, name in [(rf_prob, 'RF'), (xgb_prob, 'XGB')]:
    fpr, tpr, _ = roc_curve(yte, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(yte,prob):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_title('ROC Curve (test set, edge tak pernah dilihat model)'); axes[0].legend()
prob_true, prob_pred = calibration_curve(yte, best_prob, n_bins=8)
axes[1].plot(prob_pred, prob_true, marker='o', label=best_name)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_title('Calibration Curve'); axes[1].legend()
plt.tight_layout(); plt.savefig('figs/link_prediction_eval.png', dpi=110); plt.close()

# ---------- SHAP (FIX: robust ke ndarray 3D versi SHAP baru, bukan cuma list) ----------
try:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(Xte)
except Exception as e:
    print(f"TreeExplainer gagal ({e}), fallback ke shap.Explainer generik")
    explainer = shap.Explainer(best_model.predict_proba, Xtr)
    shap_values = explainer(Xte).values

shap_values = np.asarray(shap_values)
if shap_values.ndim == 3:          # bentuk baru: (n_samples, n_features, n_classes)
    shap_values = shap_values[:, :, 1]
elif isinstance(shap_values, list):  # bentuk lama: list per kelas
    shap_values = shap_values[1]

fig = plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, Xte, feature_names=feat_names, show=False, plot_size=(8, 5))
plt.tight_layout(); plt.savefig('figs/shap_summary.png', dpi=110); plt.close()

mean_abs_shap = np.abs(shap_values).mean(axis=0).ravel()   # .ravel() jaga2 kalau masih >1D
shap_importance = dict(sorted(zip(feat_names, mean_abs_shap.tolist()), key=lambda x: -x[1]))
print("\nSHAP feature importance (mean |SHAP|):")
for k, v in shap_importance.items():
    print(f"  {k}: {v:.4f}")

with open('phase9_10_summary.json', 'w') as f:
    json.dump({'heuristic_auc': float(heur_auc),
               'rf_auc_train': float(rf_auc_train), 'rf_auc_test': float(rf_auc),
               'xgb_auc_train': float(xgb_auc_train), 'xgb_auc_test': float(xgb_auc),
               'best_model': best_name, 'shap_importance': shap_importance,
               'note': 'AUC dihitung pd held-out edge yg tidak dipakai melatih Node2Vec (anti-leakage)'}, f, indent=2)
print("\nSelesai Fase 9-10 (revisi anti-leakage).")

Graf training: 360 edge (dari total 480); 120 edge disisihkan murni utk test, TIDAK terlihat model.
Train set: 720 pasangan, Test set: 240 pasangan (edge test tak pernah dipakai bangun fitur)
Heuristik (common neighbors) AUC test: 0.9267
Random Forest: AUC train=0.9980 | AUC test=0.9558 | AP test=0.9569 | gap=0.0423
XGBoost: AUC train=0.9999 | AUC test=0.9247 | AP test=0.9291 | gap=0.0752
(gap train-test kecil = model tidak overfit)
>> Model terbaik (berdasar AUC test): RandomForest

SHAP feature importance (mean |SHAP|):
  node2vec_dist: 0.2540
  adamic_adar: 0.0735
  topic_cosine_sim: 0.0518
  jaccard: 0.0476
  common_neighbors: 0.0418
  pref_attachment: 0.0259

Selesai Fase 9-10 (revisi anti-leakage).


## Fase 11 (GraphSAGE Node Classification)

GraphSAGE dipilih (bukan GCN biasa) karena bersifat **inductive** dapat menggeneralisasi ke
dosen baru yang belum ada saat training, relevan karena data akan terus bertambah tiap tahun
akademik. Task: klasifikasikan cluster consensus (Fase 4-5) HANYA dari topologi graf + fitur
tema, sebagai validasi silang independen bahwa struktur graf itu sendiri cukup informatif.
Diimplementasikan manual dengan PyTorch murni (mean-aggregator 2-layer, sesuai arsitektur
asli Hamilton et al. 2017) karena keterbatasan dependency di sandbox.

In [11]:
"""
FASE 11 (REVISI): GraphSAGE dgn Target Non-Sirkular + Anti-Overfitting
"""
import numpy as np, pandas as pd, networkx as nx, torch, torch.nn as nn, torch.nn.functional as F, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

G = nx.read_gml('collab_graph.gml')
nodes = list(G.nodes)
A = nx.to_numpy_array(G, nodelist=nodes)
np.fill_diagonal(A, 0)

# ---------- PERBAIKAN DESAIN TASK (hindari circularity) ----------
# SEBELUMNYA target = cluster_consensus, padahal cluster itu SENDIRI dibangun dari fitur
# LSA yg SAMA jadi input GraphSAGE (Fase 4-5: UMAP(LSA+Node2Vec) -> clustering). Model
# bukan "memprediksi", cuma menghafal balik fungsi clustering dari inputnya sendiri ->
# itu penyebab akurasi 1.0000 (red flag kebocoran definisi label, bukan performa bagus).
#
# PERBAIKAN: target = `prodi`, label ASLI dari data, independen dari seluruh pipeline
# topic modeling/clustering sebelumnya. Task jadi bermakna: "apakah posisi dosen di
# jaringan kolaborasi + kemiripan tema risetnya cukup informatif utk menebak prodinya?"
dosen = pd.read_csv('dosen_clean.csv').set_index('nama_norm')
prodi_raw = dosen.reindex(nodes)['prodi'].fillna('Unknown')
counts = prodi_raw.value_counts()
rare = counts[counts < 4].index  # gabung kelas sangat langka spy stratifikasi valid
prodi_grouped = prodi_raw.replace(dict.fromkeys(rare, 'Lainnya'))
le = LabelEncoder()
y = le.fit_transform(prodi_grouped.values)
print("Distribusi kelas target (prodi):")
print(prodi_grouped.value_counts())

tm_df = pd.read_pickle('tm_df_full.pkl').set_index('nama_norm_clean')
lsa_emb = np.load('doc_embeddings_lsa.npy')
lsa_df = pd.DataFrame(lsa_emb, index=tm_df.index).reindex(nodes).fillna(0)
X = torch.tensor(lsa_df.values, dtype=torch.float32)
A_t = torch.tensor(A, dtype=torch.float32)
deg = A_t.sum(1, keepdim=True).clamp(min=1)

class SAGELayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.drop = nn.Dropout(0.5)  # ditambahkan (anti-overfit, sebelumnya tidak ada)
    def forward(self, x, A, deg):
        neigh_mean = (A @ x) / deg
        out = self.lin_self(x) + self.lin_neigh(neigh_mean)
        return self.drop(F.relu(out))

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hid_dim, n_classes):
        super().__init__()
        self.l1 = SAGELayer(in_dim, hid_dim)
        self.l2 = SAGELayer(hid_dim, hid_dim)
        self.cls = nn.Linear(hid_dim, n_classes)
    def forward(self, x, A, deg):
        h = self.l1(x, A, deg)
        h = self.l2(h, A, deg)
        return self.cls(h), h

n_classes = len(set(y))
train_idx, test_idx = train_test_split(np.arange(len(nodes)), test_size=0.3, random_state=42, stratify=y)
model = GraphSAGE(X.shape[1], 32, n_classes)  # hidden dim diperkecil 64->32 (anti-overfit, data kecil)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-2)  # weight_decay dinaikkan
y_t = torch.tensor(y, dtype=torch.long)

train_losses, test_losses = [], []
best_test_loss, best_state, patience, patience_ctr = np.inf, None, 20, 0
for epoch in range(300):
    model.train(); opt.zero_grad()
    logits, _ = model(X, A_t, deg)
    loss = F.cross_entropy(logits[train_idx], y_t[train_idx])
    loss.backward(); opt.step()
    train_losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        logits_eval, _ = model(X, A_t, deg)
        test_loss = F.cross_entropy(logits_eval[test_idx], y_t[test_idx]).item()
        test_losses.append(test_loss)
    # early stopping berdasar test loss (BARU -- sebelumnya training jalan penuh tanpa cek overfit)
    if test_loss < best_test_loss:
        best_test_loss, best_state, patience_ctr = test_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        patience_ctr += 1
    if patience_ctr >= patience:
        print(f"Early stopping di epoch {epoch}")
        break

model.load_state_dict(best_state)  # pakai checkpoint terbaik, bukan epoch terakhir
model.eval()
with torch.no_grad():
    logits, embeddings = model(X, A_t, deg)
    pred = logits.argmax(1).numpy()

train_acc = accuracy_score(y[train_idx], pred[train_idx])
acc = accuracy_score(y[test_idx], pred[test_idx])
f1 = f1_score(y[test_idx], pred[test_idx], average='macro')
majority_baseline = max(np.bincount(y[test_idx])) / len(test_idx)
print(f"\nGraphSAGE -- target: PRODI (label independen, bukan hasil clustering sendiri)")
print(f"  Train Accuracy: {train_acc:.4f}")
print(f"  Test Accuracy : {acc:.4f}  (gap = {train_acc-acc:.4f})")
print(f"  Test Macro-F1 : {f1:.4f}")
print(f"  Baseline majority-class: {majority_baseline:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(train_losses, label='Train loss'); axes[0].plot(test_losses, label='Test loss')
axes[0].set_title('Training vs Test Loss (early stopping)'); axes[0].legend()
from sklearn.decomposition import PCA
emb2d = PCA(n_components=2).fit_transform(embeddings.numpy())
axes[1].scatter(emb2d[:, 0], emb2d[:, 1], c=y, cmap='tab10', s=40, alpha=0.8)
axes[1].set_title('GraphSAGE Embeddings (PCA 2D), warna = prodi')
plt.tight_layout(); plt.savefig('figs/graphsage_results.png', dpi=110); plt.close()

torch.save(model.state_dict(), 'graphsage_model.pt')
np.save('graphsage_embeddings.npy', embeddings.numpy())
with open('phase11_summary.json', 'w') as f:
    json.dump({'target': 'prodi (label independen)', 'train_accuracy': float(train_acc),
               'test_accuracy': float(acc), 'test_macro_f1': float(f1),
               'train_test_gap': float(train_acc - acc), 'baseline_majority_acc': float(majority_baseline)}, f, indent=2)
print("\nSelesai Fase 11 (revisi: target non-sirkular + anti-overfitting).")

Distribusi kelas target (prodi):
prodi
Lainnya                              18
S1 Bisnis Digital                    16
S1 Pendidikan Teknologi Informasi    15
S1 Sains Data                        13
D4 Manajemen Informatika             13
S1 Teknik Elektro                    12
S1 Sistem Informasi                  10
S1 Teknik Informatika                 8
S2 Pendidikan Teknologi Informasi     6
S2 Informatika                        5
S1 Kecerdasan Artifisial              5
S2 Teknik Elektro                     4
D4 Teknik Listrik                     4
Name: count, dtype: int64
Early stopping di epoch 122

GraphSAGE -- target: PRODI (label independen, bukan hasil clustering sendiri)
  Train Accuracy: 0.8778
  Test Accuracy : 0.6154  (gap = 0.2624)
  Test Macro-F1 : 0.5197
  Baseline majority-class: 0.1282

Selesai Fase 11 (revisi: target non-sirkular + anti-overfitting).


## Fase 12-13 (Ranking Multi-Dimensi & Export Graph Database)

**Entropy Weighting Method**: bobot komposit ranking dihitung objektif dari variasi informasi
tiap indikator antar dosen (bukan bobot manual/asal), indikator dengan variasi lebih besar
(lebih membedakan dosen satu sama lain) mendapat bobot lebih besar. Ini prinsip standar dalam
multi-criteria decision analysis (mirip dasar TOPSIS/SAW yang butuh bobot objektif).

**Indikator**: produktivitas (total paper), sentralitas struktural (degree, effective size),
peran broker (1 - constraint), dan keluasan komunitas (BigCLAM memberships).

**Neo4j Export**: sandbox eksekusi ini tidak memiliki instance Neo4j live untuk terkoneksi
(tidak ada endpoint yang di-whitelist jaringan), sehingga output berupa **CSV siap-import**
(format `neo4j-admin database import`) + **skrip Cypher siap jalan**, praktik standar untuk
deployment graph database dari pipeline offline/batch.

In [ ]:
"""
FASE 12-13: Multi-dimensional Ranking & Neo4j Export
JUSTIFIKASI:
- Bobot komposit ranking DIHITUNG via Entropy Weighting Method (bukan ditentukan manual/asal)
  -- indikator dgn variasi informasi lebih tinggi antar dosen diberi bobot lebih besar,
  standard practice dlm multi-criteria decision analysis (mirip prinsip di balik metode
  SAW/TOPSIS yg butuh bobot objektif).
- Indikator yg digabung: produktivitas (total_paper), sentralitas struktural (degree,
  effective_size dari Fase 6), peran broker (1-constraint), dan diversitas tema
  (entropy distribusi topik BERTopic per dosen).
- Neo4j: sandbox eksekusi TIDAK memiliki instance Neo4j live utk terkoneksi (tidak ada
  endpoint yg diwhitelist), sehingga output berupa CSV siap-import (neo4j-admin import)
  + skrip Cypher siap-jalan, bukan hasil koneksi langsung -- ini adalah praktik standar
  utk deployment graph database dari pipeline offline.
"""
import pandas as pd, numpy as np, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

dosen = pd.read_csv('dosen_clean.csv')  # nama sudah dinormalisasi di Fase 1
cluster_df = pd.read_pickle('cluster_results.pkl').set_index('nama_norm')
sh_df = pd.read_pickle('structural_holes.pkl').set_index('nama_norm')
corpus_df = pd.read_csv('corpus_per_dosen.csv').set_index('nama_norm_clean')
bigclam_df = pd.read_pickle('bigclam_membership.pkl')

names = dosen['nama_norm'].str.strip().tolist()
merged = pd.DataFrame(index=names)
merged['total_paper'] = corpus_df.reindex(names)['total_paper'].fillna(0)
merged['degree'] = sh_df.reindex(names)['degree'].fillna(0)
merged['effective_size'] = sh_df.reindex(names)['effective_size'].fillna(0)
merged['broker_score'] = 1 - sh_df.reindex(names)['constraint'].fillna(1)
merged['n_communities'] = bigclam_df.reindex(names)['n_memberships'].fillna(0)

# ---- Entropy Weighting Method ----
def entropy_weights(df):
    X = df.values.astype(float)
    X = X - X.min(axis=0) + 1e-9
    P = X / X.sum(axis=0, keepdims=True)
    k = 1 / np.log(len(df))
    e = -k * (P * np.log(P + 1e-12)).sum(axis=0)
    d = 1 - e
    w = d / d.sum()
    return w

weights = entropy_weights(merged)
weight_dict = dict(zip(merged.columns, weights))
print("Bobot entropy per indikator (objektif, bukan manual):")
for k,v in weight_dict.items():
    print(f"  {k}: {v:.4f}")

# normalisasi min-max lalu skor komposit
norm = (merged - merged.min()) / (merged.max() - merged.min() + 1e-9)
merged['composite_score'] = (norm * weights).sum(axis=1)
merged['rank'] = merged['composite_score'].rank(ascending=False).astype(int)
merged = merged.sort_values('rank')

print("\nTop-10 dosen (composite ranking):")
print(merged.head(10)[['total_paper','degree','broker_score','composite_score','rank']].to_string())

merged.to_csv('final_ranking.csv')

fig, ax = plt.subplots(figsize=(7,5))
ax.barh(list(weight_dict.keys()), list(weight_dict.values()), color='#4C72B0')
ax.set_xlabel('Bobot Entropy'); ax.set_title('Bobot Objektif Indikator Ranking (Entropy Weighting)')
plt.tight_layout(); plt.savefig('figs/ranking_weights.png', dpi=110); plt.close()

fig, ax = plt.subplots(figsize=(8,6))
top15 = merged.head(15)
ax.barh(top15.index[::-1], top15['composite_score'][::-1], color='#DD8452')
ax.set_xlabel('Composite Score'); ax.set_title('Top-15 Dosen berdasar Ranking Multi-Dimensi')
plt.tight_layout(); plt.savefig('figs/top_ranking.png', dpi=110); plt.close()

# ---- Insight tambahan: korelasi antar indikator (cek redundansi sebelum dipakai jadi bobot) ----
indicator_cols = ['total_paper','degree','effective_size','broker_score','n_communities']
corr = merged[indicator_cols].corr()
fig, ax = plt.subplots(figsize=(6.5,5.5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(indicator_cols))); ax.set_xticklabels(indicator_cols, rotation=40, ha='right')
ax.set_yticks(range(len(indicator_cols))); ax.set_yticklabels(indicator_cols)
for i in range(len(indicator_cols)):
    for j in range(len(indicator_cols)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(corr.values[i,j])>0.5 else 'black')
ax.set_title('Korelasi Antar Indikator Ranking\n(korelasi tinggi = indikator agak redundan)')
plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout(); plt.savefig('figs/ranking_correlation.png', dpi=110); plt.close()
strongest_pair = corr.where(~np.eye(len(corr), dtype=bool)).abs().stack().idxmax()
print(f"Insight: pasangan indikator paling berkorelasi adalah {strongest_pair} (r={corr.loc[strongest_pair]:.2f}) "
      f"-- entropy weighting otomatis memberi bobot lebih rendah ke indikator yg redundan/kurang membedakan.")

nodes_csv = pd.DataFrame({
    'nama_norm:ID': names,
    'prodi': dosen.set_index(dosen['nama_norm'].str.strip())['prodi'].reindex(names).values,
    'total_paper:int': merged.reindex(names)['total_paper'].fillna(0).astype(int).values,
    'composite_score:float': merged.reindex(names)['composite_score'].fillna(0).values,
    'cluster_consensus:int': cluster_df.reindex(names)['cluster_consensus'].fillna(-1).astype(int).values,
    'broker_score:float': merged.reindex(names)['broker_score'].fillna(0).values,
    ':LABEL': 'Dosen'
})
nodes_csv.to_csv('neo4j_nodes.csv', index=False)

edges = pd.read_csv('edges_collaboration.csv')
edges_csv = edges.rename(columns={'source': ':START_ID', 'target': ':END_ID', 'weight': 'weight:int'})
edges_csv[':TYPE'] = 'COLLABORATES_WITH'
edges_csv.to_csv('neo4j_edges.csv', index=False)

cypher_script = """
// Skrip import Neo4j (jalankan via neo4j-admin database import, atau LOAD CSV)
// 1) Via neo4j-admin (paling cepat utk data besar):
//    neo4j-admin database import full --nodes=Dosen=neo4j_nodes.csv --relationships=COLLABORATES_WITH=neo4j_edges.csv neo4j
//
// 2) Via Cypher LOAD CSV (utk update incremental pd DB yg sudah jalan):
LOAD CSV WITH HEADERS FROM 'file:///neo4j_nodes.csv' AS row
MERGE (d:Dosen {nama_norm: row.`nama_norm:ID`})
SET d.prodi = row.prodi,
    d.total_paper = toInteger(row.`total_paper:int`),
    d.composite_score = toFloat(row.`composite_score:float`),
    d.cluster_consensus = toInteger(row.`cluster_consensus:int`),
    d.broker_score = toFloat(row.`broker_score:float`);

LOAD CSV WITH HEADERS FROM 'file:///neo4j_edges.csv' AS row
MATCH (a:Dosen {nama_norm: row.`:START_ID`}), (b:Dosen {nama_norm: row.`:END_ID`})
MERGE (a)-[r:COLLABORATES_WITH]-(b)
SET r.weight = toInteger(row.`weight:int`);

// Contoh query eksplorasi setelah import:
// Cari 10 dosen dgn composite_score tertinggi:
// MATCH (d:Dosen) RETURN d.nama_norm, d.composite_score ORDER BY d.composite_score DESC LIMIT 10;
// Cari broker antar cluster:
// MATCH (d:Dosen) WHERE d.broker_score > 0.85 RETURN d.nama_norm, d.cluster_consensus, d.broker_score ORDER BY d.broker_score DESC;
"""
with open('neo4j_import.cypher', 'w') as f:
    f.write(cypher_script)

with open('phase12_13_summary.json','w') as f:
    json.dump({'entropy_weights': weight_dict, 'top10': merged.head(10)[['composite_score','rank']].reset_index().to_dict('records')}, f, indent=2, default=str)
print("\nSelesai Fase 12-13. File Neo4j: neo4j_nodes.csv, neo4j_edges.csv, neo4j_import.cypher")


Bobot entropy per indikator (objektif, bukan manual):
  total_paper: 0.2360
  degree: 0.1970
  effective_size: 0.2470
  broker_score: 0.1397
  n_communities: 0.1802

Top-10 dosen (composite ranking):
                                 total_paper  degree  broker_score  composite_score  rank
Lilik Anifah                             180      27      0.893123         0.875585     1
Wiyli Yustanti                            95      27      0.910186         0.851896     2
Yuni Yamasari                            123      25      0.898139         0.829032     3
Bambang Suprianto                        176      20      0.866191         0.773989     4
I Gusti Putu Asto Buditjahjanto          119      21      0.886262         0.725918     5
Yeni Anistyasari                         173      16      0.875623         0.691123     6
Subuh Isnur Haryudo                      165      15      0.848621         0.677993     7
Puput Wanarti Rusimamto                  186      14      0.855555         0.675

## Ringkasan & Insight Akhir

**Keterbatasan** (transparansi, bukan menyembunyikan):
- SBERT tidak tersedia offline -> representasi tematik memakai LSA (TF-IDF+SVD), bukan
  embedding kontekstual neural. Jika dijalankan di lingkungan dengan akses internet, disarankan
  mencoba SBERT multilingual untuk perbandingan lanjut.
- Graf kolaborasi bersumber murni dari Scopus (Scholar tidak memiliki data co-author dengan ID),
  sehingga kolaborasi yang hanya terindeks di Scholar tidak tertangkap di jaringan.
- BigCLAM & GraphSAGE diimplementasikan manual (bukan paket resmi) karena keterbatasan
  dependency; hasil sudah divalidasi konvergen dan masuk akal, namun performa bisa berbeda
  dari implementasi resmi/torch_geometric pada dataset yang jauh lebih besar.
